Phase 5.4 — final retrain on train+val merged. Reads the best (dropout, L2) winner from 05c (`deepfm_sweep_dropout_l2.json`), fixes `emb_dim=32` from 05b Stage A, and trains one DeepFM v2 (with `item_text_emb_pca32`) on the union of train + val for the same number of epochs the winning config peaked at. The result is the final ranker that Phase 7 evaluates against test (test is locked, evaluated exactly once).

In [1]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import platform

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
MODELS_DIR = PROJECT_ROOT / "models"

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"arch: {platform.machine()}  device: {DEVICE}")

arch: arm64  device: mps


Inline IDEncoder + Dataset (with v2 item_text) + DeepFM v2 — same code as 05c.

In [2]:
class IDEncoder:
    def __init__(self, ids, oov_token="<UNK>"):
        oov_markers = {"<NEW_USER>", "<NEW_BUSINESS>", "<UNK>", oov_token}
        unique_real_ids = sorted({i for i in ids if i not in oov_markers})
        self.id_to_idx = {oov_token: 0}
        for idx, _id in enumerate(unique_real_ids, start=1):
            self.id_to_idx[_id] = idx
        for marker in oov_markers:
            self.id_to_idx.setdefault(marker, 0)
        self._size = 1 + len(unique_real_ids)
    def __len__(self): return self._size
    def encode(self, _id): return self.id_to_idx.get(_id, 0)
    def encode_array(self, ids): return ids.map(self.id_to_idx).fillna(0).astype(np.int64).values


class TasteHunterDataset(Dataset):
    def __init__(self, df, user_encoder, item_encoder, user_features, item_features):
        self.user_idx = torch.from_numpy(user_encoder.encode_array(df["user_id"]))
        self.item_idx = torch.from_numpy(item_encoder.encode_array(df["business_id"]))
        self.label = torch.from_numpy(df["label"].astype(np.float32).values)
        n_users = len(user_encoder); n_items = len(item_encoder)
        self.user_num = np.zeros((n_users, 6), dtype=np.float32)
        self.user_cuisine = np.zeros((n_users, 50), dtype=np.float32)
        for _, row in user_features.iterrows():
            uidx = user_encoder.encode(row["user_id"])
            self.user_num[uidx] = [row["avg_rating_given"], row["review_count_log"], row["days_active"],
                                    row["elite_flag"], row["mean_distance_traveled"], row["price_tolerance_avg"]]
            emb = row["fav_cuisine_emb"]
            if isinstance(emb, list) and len(emb) == 50:
                self.user_cuisine[uidx] = emb
        self.item_num = np.zeros((n_items, 7), dtype=np.float32)
        self.item_cat = np.zeros((n_items, 50), dtype=np.float32)
        self.item_text = np.zeros((n_items, 32), dtype=np.float32)
        for _, row in item_features.iterrows():
            iidx = item_encoder.encode(row["business_id"])
            self.item_num[iidx] = [row["avg_rating"], row["review_count_log"], row["price_level"],
                                    row["is_open"], row["has_outdoor_seating"], row["photo_count_log"], row["city_id"]]
            cat = row["categories_multi_hot"]
            if isinstance(cat, list) and len(cat) == 50:
                self.item_cat[iidx] = cat
            text = row.get("item_text_emb_pca32")
            if isinstance(text, (list, np.ndarray)) and len(text) == 32:
                self.item_text[iidx] = np.asarray(text, dtype=np.float32)
        self.user_num_t = torch.from_numpy(self.user_num)
        self.user_cuisine_t = torch.from_numpy(self.user_cuisine)
        self.item_num_t = torch.from_numpy(self.item_num)
        self.item_cat_t = torch.from_numpy(self.item_cat)
        self.item_text_t = torch.from_numpy(self.item_text)
    def __len__(self): return len(self.label)
    def __getitem__(self, idx):
        u = self.user_idx[idx]; i = self.item_idx[idx]
        return {
            "user_idx": u, "item_idx": i, "label": self.label[idx],
            "user_num": self.user_num_t[u], "user_cuisine": self.user_cuisine_t[u],
            "item_num": self.item_num_t[i], "item_cat": self.item_cat_t[i],
            "item_text": self.item_text_t[i],
        }


class DeepFM(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=32,
                 user_num_dim=6, user_cuisine_dim=50,
                 item_num_dim=7, item_cat_dim=50, item_text_dim=32,
                 dnn_hidden=(256, 128, 64), dropout=0.2, ablate_user_id=False):
        super().__init__()
        self.emb_dim = emb_dim
        self.ablate_user_id = ablate_user_id
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        self.user_num_proj = nn.Linear(user_num_dim, emb_dim)
        self.item_num_proj = nn.Linear(item_num_dim, emb_dim)
        self.item_text_proj = nn.Linear(item_text_dim, emb_dim)
        feat_dim_linear = user_num_dim + item_num_dim + user_cuisine_dim + item_cat_dim + item_text_dim
        self.linear_features = nn.Linear(feat_dim_linear, 1)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))
        dnn_input_dim = emb_dim * 5 + user_cuisine_dim + item_cat_dim + item_text_dim
        layers = []; prev = dnn_input_dim
        for h in dnn_hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]; prev = h
        layers.append(nn.Linear(prev, 1))
        self.dnn = nn.Sequential(*layers)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_idx, item_idx, user_num, user_cuisine, item_num, item_cat, item_text):
        u_emb = self.user_emb(user_idx)
        i_emb = self.item_emb(item_idx)
        un_emb = self.user_num_proj(user_num)
        in_emb = self.item_num_proj(item_num)
        it_emb = self.item_text_proj(item_text)
        if self.ablate_user_id:
            u_emb = torch.zeros_like(u_emb)
        feat_concat = torch.cat([user_num, item_num, user_cuisine, item_cat, item_text], dim=-1)
        order1 = self.linear_features(feat_concat).squeeze(-1)
        bu = self.user_bias(user_idx).squeeze(-1)
        bi = self.item_bias(item_idx).squeeze(-1)
        if self.ablate_user_id:
            bu = torch.zeros_like(bu)
        embs = torch.stack([u_emb, i_emb, un_emb, in_emb, it_emb], dim=1)
        sum_sq = embs.sum(dim=1) ** 2
        sq_sum = (embs ** 2).sum(dim=1)
        order2 = 0.5 * (sum_sq - sq_sum).sum(dim=-1)
        dnn_input = torch.cat([u_emb, i_emb, un_emb, in_emb, it_emb, user_cuisine, item_cat, item_text], dim=-1)
        dnn_output = self.dnn(dnn_input).squeeze(-1)
        return torch.sigmoid(order1 + order2 + dnn_output + bu + bi + self.global_bias)

**Read best config from 05c sweep**, then load train + val merged and train one DeepFM with that config.

In [3]:
with open(MODELS_DIR / "deepfm_sweep_dropout_l2.json") as f:
    grid_results = json.load(f)
best_config = max(grid_results, key=lambda x: x["best_ndcg10"])
print(f"best from 05c: emb={best_config['emb_dim']} dropout={best_config['dropout']} l2={best_config['l2']:.0e}")
print(f"  best NDCG@10 (val): {best_config['best_ndcg10']:.4f}")

# Load best epoch from that config's history (so we know how many epochs to train final)
tag = best_config["tag"]
with open(MODELS_DIR / f"deepfm_{tag}_history.json") as f:
    h = json.load(f)
best_epoch = int(np.argmax(h["val_ndcg10"])) + 1
print(f"  best epoch in 05c run: {best_epoch}")

# Load train + val merged
user_features = pd.read_parquet(FEATURES_DIR / "user_features.parquet")
item_features = pd.read_parquet(FEATURES_DIR / "item_features.parquet")
train_pos_neg = pd.read_parquet(FEATURES_DIR / "train_with_negatives.parquet")
val_df = pd.read_parquet(CLEANED_DIR / "val_reviews.parquet")

# Generate val negatives (1:4 same-city) so val rows look the same as train
rng = np.random.default_rng(SEED)
biz_city = item_features.set_index("business_id")["city"].to_dict()
city_biz = {}
for bid, c in biz_city.items():
    if c == "<UNK>": continue
    city_biz.setdefault(c, []).append(bid)
for c in city_biz: city_biz[c] = np.array(city_biz[c])

val_pos = val_df[val_df["stars"] >= 4].copy()
val_pos["city"] = val_pos["business_id"].map(biz_city).fillna("<UNK>")
val_pos["label"] = 1
val_user_visited = val_df.groupby("user_id")["business_id"].agg(set).to_dict()

val_neg_rows = []
for row in val_pos.itertuples(index=False):
    if row.city not in city_biz: continue
    cands = city_biz[row.city]
    visited = val_user_visited.get(row.user_id, set())
    sampled = []
    for bid in rng.choice(cands, size=8, replace=True):
        if bid not in visited and bid != row.business_id:
            sampled.append(bid)
            if len(sampled) >= 4: break
    for bid in sampled:
        val_neg_rows.append({"user_id": row.user_id, "business_id": bid, "label": 0,
                             "stars": np.nan, "date": row.date, "city": row.city})
val_neg = pd.DataFrame(val_neg_rows)

val_with_neg = pd.concat([
    val_pos[["user_id", "business_id", "label", "stars", "date", "city"]],
    val_neg
], ignore_index=True)
print(f"val with negatives: {len(val_with_neg)} ({(val_with_neg['label']==1).sum()} pos + {(val_with_neg['label']==0).sum()} neg)")

merged = pd.concat([train_pos_neg, val_with_neg], ignore_index=True)
print(f"merged train+val: {len(merged)} samples")

best from 05c: emb=32 dropout=0.1 l2=1e-04
  best NDCG@10 (val): 0.3255
  best epoch in 05c run: 10


val with negatives: 302410 (60482 pos + 241928 neg)
merged train+val: 2377695 samples


Build dataset, encoder, train one config (best from 05c) for `best_epoch` epochs.

In [4]:
user_encoder = IDEncoder(user_features["user_id"].tolist(), oov_token="<NEW_USER>")
item_encoder = IDEncoder(item_features["business_id"].tolist(), oov_token="<NEW_BUSINESS>")
n_users, n_items = len(user_encoder), len(item_encoder)

t0 = time.time()
final_ds = TasteHunterDataset(merged, user_encoder, item_encoder, user_features, item_features)
print(f"final dataset: {len(final_ds)} samples ({time.time()-t0:.1f}s)")
final_loader = DataLoader(final_ds, batch_size=8192, shuffle=True, num_workers=0)

torch.manual_seed(SEED)
model = DeepFM(n_users, n_items, emb_dim=best_config["emb_dim"], dropout=best_config["dropout"]).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=best_config["l2"])
bce = nn.BCELoss()

history = {"epoch": [], "train_loss": []}
for epoch in range(1, best_epoch + 1):
    model.train()
    loss_sum, n_batch = 0.0, 0
    t0 = time.time()
    for batch in final_loader:
        u = batch["user_idx"].to(DEVICE)
        i = batch["item_idx"].to(DEVICE)
        y = batch["label"].to(DEVICE)
        kwargs = {k: batch[k].to(DEVICE) for k in ("user_num", "user_cuisine", "item_num", "item_cat", "item_text")}
        pred = model(u, i, **kwargs)
        loss = bce(pred, y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        loss_sum += loss.item(); n_batch += 1
    train_loss = loss_sum / n_batch
    elapsed = time.time() - t0
    print(f"epoch {epoch:02d} | {elapsed:.0f}s | loss={train_loss:.4f}")
    history["epoch"].append(epoch); history["train_loss"].append(train_loss)

torch.save(model.state_dict(), MODELS_DIR / "deepfm_final_v2.pt")
final_meta = {
    "config": best_config,
    "best_epoch_from_5c": best_epoch,
    "trained_on": "train + val merged",
    "history": history,
}
with open(MODELS_DIR / "deepfm_final_v2_meta.json", "w") as f:
    json.dump(final_meta, f, indent=2, default=str)
print(f"\nfinal v2 model saved: deepfm_final_v2.pt")

/var/folders/4j/cfzmxp0d3db8xdcx2fncnbnc0000gn/T/ipykernel_24911/2078872190.py:18: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  self.user_idx = torch.from_numpy(user_encoder.encode_array(df["user_id"]))


final dataset: 2377695 samples (4.7s)


epoch 01 | 31s | loss=0.6312


epoch 02 | 29s | loss=0.3263


epoch 03 | 30s | loss=0.3109


...

epoch 06 | 30s | loss=0.2808


epoch 07 | 30s | loss=0.2745


epoch 08 | 30s | loss=0.2689


epoch 09 | 30s | loss=0.2629


epoch 10 | 30s | loss=0.2573

final v2 model saved: deepfm_final_v2.pt


This is the model that Phase 7 evaluates on test. **Test is locked from this point** — any new training touching test data invalidates the evaluation. The same procedure can be run separately for v1 (no text_emb) by changing the DeepFM class definition and re-saving as `deepfm_final_v1.pt`.